# no-relu-on-final-layer — worked example 2: detect a stray final activation from output stats

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-relu-on-final-layer`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

On standard-normal input through well-initialized layers, an unclipped final layer produces roughly zero-mean outputs, so about half are negative. A model whose outputs are essentially never negative almost certainly has a clipping activation on its final layer.

## Worked solution

We write `has_final_relu(model, in_features)` that puts the model in eval mode, samples standard-normal input under `t.no_grad()`, runs the model, and computes the fraction of negative outputs. If that fraction is below a tiny threshold (`1e-3`), the outputs are essentially never negative, which signals a clipping activation, so we return `True`; otherwise `False`. We restore the model's original training mode afterward. We build one model with a final ReLU and one without, run the detector on both, and print the two verdicts, which come out `True` and `False`.

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(1)

class WithFinalReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8, 4)
    def forward(self, x):
        return F.relu(self.fc(x))

class NoFinalReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8, 4)
    def forward(self, x):
        return self.fc(x)

def has_final_relu(model, in_features, batch=256):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            y = model(t.randn(batch, in_features))
            frac_neg = (y < 0).float().mean().item()
    finally:
        if was_training:
            model.train()
    return frac_neg < 1e-3

print('with relu:', has_final_relu(WithFinalReLU(), 8))
print('no relu:', has_final_relu(NoFinalReLU(), 8))